# Delivery Performance and Customer Retention at a Brazilian Marketplace
**Dataset:** Olist Brazilian E-Commerce Public Dataset (Kaggle, ~100k orders, 2016-2018)

## 1. Problem statement
The Head of Operations at an online marketplace wants to know whether **late deliveries are hurting customer satisfaction and repeat purchases**, which **regions, categories and sellers** are causing the problem, and **what should be fixed first** to protect revenue.

**Business question:** How much revenue and customer loyalty is the business losing to delivery delays, and where should it act first?

**Setup:** download the dataset from Kaggle (`olistbr/brazilian-ecommerce`), unzip it, and set `DATA_DIR` below to the folder containing the CSV files.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 5)
pd.options.display.float_format = "{:,.2f}".format

DATA_DIR = "./olist_data"   # <-- change to your folder

## 2. Load data

In [ ]:
def load(name, **kw):
    return pd.read_csv(f"{DATA_DIR}/{name}", **kw)

orders = load("olist_orders_dataset.csv", parse_dates=[
    "order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
    "order_delivered_customer_date", "order_estimated_delivery_date"])
items = load("olist_order_items_dataset.csv")
reviews = load("olist_order_reviews_dataset.csv")
customers = load("olist_customers_dataset.csv")
sellers = load("olist_sellers_dataset.csv")
products = load("olist_products_dataset.csv")
translation = load("product_category_name_translation.csv")
geo = load("olist_geolocation_dataset.csv")

for name, df in [("orders", orders), ("items", items), ("reviews", reviews), ("customers", customers),
                 ("sellers", sellers), ("products", products), ("geolocation", geo)]:
    print(f"{name:12s} {df.shape}")

## 3. Data quality check

In [ ]:
print("Order status counts:")
print(orders["order_status"].value_counts())
print("\nMissing values in orders:")
print(orders.isna().sum()[lambda s: s > 0])
print("\nDuplicate order_ids:", orders["order_id"].duplicated().sum())

**Cleaning decisions**
- Delivery metrics use only orders with status `delivered` and a delivery date.
- Some orders have several reviews and several items, so both are aggregated to one row per order before joining (prevents double counting revenue).
- Category names are translated to English; missing categories become `unknown`.

## 4. Build the analysis tables

In [ ]:
# Category in English
products = products.merge(translation, on="product_category_name", how="left")
products["category"] = products["product_category_name_english"].fillna("unknown")

# Item-level table (used for category and seller analysis)
items_full = (items
    .merge(products[["product_id", "category"]], on="product_id", how="left")
    .merge(sellers[["seller_id", "seller_state", "seller_zip_code_prefix"]], on="seller_id", how="left"))
items_full["category"] = items_full["category"].fillna("unknown")
items_full["item_revenue"] = items_full["price"] + items_full["freight_value"]

# One row per order: revenue, freight, item count
order_items = items_full.groupby("order_id").agg(
    revenue=("item_revenue", "sum"),
    freight=("freight_value", "sum"),
    n_items=("order_item_id", "count"),
    n_sellers=("seller_id", "nunique")).reset_index()

# One row per order: average review score
order_reviews = reviews.groupby("order_id").agg(review_score=("review_score", "mean")).reset_index()

# Master order table
df = (orders
    .merge(customers[["customer_id", "customer_unique_id", "customer_state", "customer_zip_code_prefix"]], on="customer_id", how="left")
    .merge(order_items, on="order_id", how="left")
    .merge(order_reviews, on="order_id", how="left"))

df["purchase_month"] = df["order_purchase_timestamp"].dt.to_period("M").dt.to_timestamp()
print(df.shape)
df.head()

## 5. Feature engineering

In [ ]:
delivered = df[(df["order_status"] == "delivered") & df["order_delivered_customer_date"].notna()].copy()

delivered["delivery_days"] = (delivered["order_delivered_customer_date"] - delivered["order_purchase_timestamp"]).dt.days
delivered["delay_days"] = (delivered["order_delivered_customer_date"] - delivered["order_estimated_delivery_date"]).dt.total_seconds() / 86400
delivered["on_time"] = delivered["delay_days"] <= 0

def bucket(d):
    if d <= 0: return "On time"
    if d <= 3: return "1-3 days late"
    if d <= 7: return "4-7 days late"
    return "8+ days late"

delivered["delay_bucket"] = delivered["delay_days"].apply(bucket)
bucket_order = ["On time", "1-3 days late", "4-7 days late", "8+ days late"]
delivered["delay_bucket"] = pd.Categorical(delivered["delay_bucket"], categories=bucket_order, ordered=True)

# Sanity filter: remove impossible delivery times
delivered = delivered[(delivered["delivery_days"] >= 0) & (delivered["delivery_days"] < 120)]
print("Delivered orders analysed:", len(delivered))
delivered[["delivery_days", "delay_days", "on_time", "delay_bucket"]].head()

## 6. KPIs

In [ ]:
total_orders = len(df)
kpi = {
    "Total revenue (BRL)": df["revenue"].sum(),
    "Total orders": total_orders,
    "Average order value (BRL)": df["revenue"].mean(),
    "On-time delivery rate": delivered["on_time"].mean(),
    "Average delivery time (days)": delivered["delivery_days"].mean(),
    "Average delay of late orders (days)": delivered.loc[~delivered["on_time"], "delay_days"].mean(),
    "Average review score": delivered["review_score"].mean(),
    "Cancellation rate": (df["order_status"] == "canceled").mean(),
}

cust_orders = df[df["order_status"] != "canceled"].groupby("customer_unique_id")["order_id"].nunique()
kpi["Repeat purchase rate"] = (cust_orders > 1).mean()

kpi_df = pd.DataFrame(kpi.items(), columns=["KPI", "Value"]).set_index("KPI")
kpi_df

## 7. Trends

In [ ]:
monthly = (df[df["order_status"] != "canceled"].groupby("purchase_month")
    .agg(revenue=("revenue", "sum"), orders=("order_id", "count")).reset_index())
monthly_q = (delivered.groupby("purchase_month")
    .agg(on_time_rate=("on_time", "mean"), review=("review_score", "mean"), n=("order_id", "count")).reset_index())
monthly = monthly.merge(monthly_q, on="purchase_month", how="left")
monthly = monthly[monthly["orders"] >= 100]   # drop sparse 2016 months

fig, ax = plt.subplots(1, 2, figsize=(15, 4.5))
ax[0].plot(monthly["purchase_month"], monthly["revenue"] / 1e6, marker="o")
ax[0].set_title("Monthly revenue (million BRL)")
ax[1].plot(monthly["purchase_month"], monthly["orders"], marker="o", color="tab:green")
ax[1].set_title("Monthly orders")
for a in ax: a.tick_params(axis="x", rotation=45)
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(1, 2, figsize=(15, 4.5))
ax[0].plot(monthly["purchase_month"], monthly["on_time_rate"] * 100, marker="o", color="tab:red")
ax[0].set_title("On-time delivery rate (%)")
ax[1].plot(monthly["purchase_month"], monthly["review"], marker="o", color="tab:purple")
ax[1].set_title("Average review score")
for a in ax: a.tick_params(axis="x", rotation=45)
plt.tight_layout(); plt.show()

peak = monthly.loc[monthly["revenue"].idxmax()]
print(f"Peak revenue month: {peak['purchase_month']:%b %Y} ({peak['revenue']:,.0f} BRL)")

## 8. Drivers

### 8.1 Delay vs customer satisfaction

In [ ]:
by_bucket = (delivered.dropna(subset=["review_score"]).groupby("delay_bucket")
    .agg(avg_review=("review_score", "mean"), orders=("order_id", "count"),
         pct_1_2_star=("review_score", lambda s: (s <= 2).mean())).reset_index())
display(by_bucket)

fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
sns.barplot(data=by_bucket, x="delay_bucket", y="avg_review", ax=ax[0])
ax[0].set_title("Average review score by delivery delay"); ax[0].set_ylim(1, 5)
sns.barplot(data=by_bucket, x="delay_bucket", y="pct_1_2_star", ax=ax[1], color="tab:red")
ax[1].set_title("Share of 1-2 star reviews by delivery delay")
plt.tight_layout(); plt.show()

### 8.2 Delay by customer state

In [ ]:
state = (delivered.groupby("customer_state")
    .agg(orders=("order_id", "count"), revenue=("revenue", "sum"),
         on_time_rate=("on_time", "mean"), avg_delivery_days=("delivery_days", "mean"),
         avg_review=("review_score", "mean")).reset_index())
state_big = state[state["orders"] >= 300].sort_values("on_time_rate")

fig, ax = plt.subplots(1, 2, figsize=(15, 6))
sns.barplot(data=state_big, y="customer_state", x="on_time_rate", ax=ax[0], color="tab:blue")
ax[0].set_title("On-time rate by state (states with 300+ orders)")
sns.barplot(data=state_big, y="customer_state", x="avg_delivery_days", ax=ax[1], color="tab:orange")
ax[1].set_title("Average delivery days by state")
plt.tight_layout(); plt.show()
state_big.head(10)

### 8.3 Does distance explain delays?

In [ ]:
# Average coordinates per zip prefix
zip_geo = geo.groupby("geolocation_zip_code_prefix").agg(lat=("geolocation_lat", "mean"), lng=("geolocation_lng", "mean")).reset_index()

def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dphi = p2 - p1
    dl = np.radians(lon2) - np.radians(lon1)
    a = np.sin(dphi / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dl / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))

# Use the first item's seller as the order's seller
first_item = items_full.sort_values("order_item_id").drop_duplicates("order_id")[["order_id", "seller_zip_code_prefix", "seller_id"]]
dist = (delivered[["order_id", "customer_zip_code_prefix", "delay_days", "delivery_days", "on_time", "review_score"]]
    .merge(first_item, on="order_id")
    .merge(zip_geo.rename(columns={"geolocation_zip_code_prefix": "customer_zip_code_prefix", "lat": "c_lat", "lng": "c_lng"}), on="customer_zip_code_prefix")
    .merge(zip_geo.rename(columns={"geolocation_zip_code_prefix": "seller_zip_code_prefix", "lat": "s_lat", "lng": "s_lng"}), on="seller_zip_code_prefix"))
dist["distance_km"] = haversine(dist["s_lat"], dist["s_lng"], dist["c_lat"], dist["c_lng"])
dist["distance_band"] = pd.cut(dist["distance_km"], [0, 100, 500, 1000, 2000, 5000], labels=["<100 km", "100-500", "500-1000", "1000-2000", "2000+"])

band = dist.groupby("distance_band").agg(orders=("order_id", "count"), avg_delivery_days=("delivery_days", "mean"),
                                         on_time_rate=("on_time", "mean"), avg_review=("review_score", "mean")).reset_index()
display(band)
print("Correlation (distance vs delivery days):", round(dist["distance_km"].corr(dist["delivery_days"]), 2))

fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
sns.barplot(data=band, x="distance_band", y="avg_delivery_days", ax=ax[0]); ax[0].set_title("Delivery days by distance")
sns.barplot(data=band, x="distance_band", y="on_time_rate", ax=ax[1], color="tab:red"); ax[1].set_title("On-time rate by distance")
plt.tight_layout(); plt.show()

### 8.4 Delay by product category

In [ ]:
item_del = items_full.merge(delivered[["order_id", "on_time", "delay_days", "review_score"]], on="order_id")
cat = (item_del.groupby("category")
    .agg(revenue=("item_revenue", "sum"), orders=("order_id", "nunique"),
         on_time_rate=("on_time", "mean"), avg_review=("review_score", "mean")).reset_index())
cat_big = cat[cat["orders"] >= 500].copy()

top_cat = cat_big.sort_values("revenue", ascending=False).head(15)
fig, ax = plt.subplots(figsize=(11, 6))
sns.barplot(data=top_cat.sort_values("on_time_rate"), y="category", x="on_time_rate", color="tab:blue", ax=ax)
ax.set_title("On-time rate for the top-15 revenue categories")
plt.tight_layout(); plt.show()

## 9. Customer retention

In [ ]:
# Customer's first order and whether it was late
cust = df[df["order_status"] == "delivered"].sort_values("order_purchase_timestamp")
n_orders = cust.groupby("customer_unique_id")["order_id"].nunique().rename("n_orders")
first = cust.drop_duplicates("customer_unique_id")[["customer_unique_id", "order_id"]].merge(
    delivered[["order_id", "on_time", "delay_bucket", "review_score"]], on="order_id").merge(n_orders, on="customer_unique_id")
first["repeat"] = first["n_orders"] > 1

repeat_by_bucket = first.groupby("delay_bucket").agg(customers=("repeat", "size"), repeat_rate=("repeat", "mean")).reset_index()
display(repeat_by_bucket)

first["low_review"] = first["review_score"] <= 2
repeat_by_review = first.dropna(subset=["review_score"]).groupby("low_review").agg(customers=("repeat", "size"), repeat_rate=("repeat", "mean"))
display(repeat_by_review)

sns.barplot(data=repeat_by_bucket, x="delay_bucket", y="repeat_rate")
plt.title("Repeat purchase rate by first-order delivery outcome"); plt.show()

## 10. Risks

In [ ]:
# Revenue concentration by state
st_rev = df[df["order_status"] != "canceled"].groupby("customer_state")["revenue"].sum().sort_values(ascending=False)
share = (st_rev / st_rev.sum())
print("Top 3 states share of revenue:", round(share.head(3).sum() * 100, 1), "%")
share.head(8).mul(100).round(1).rename("% of revenue").to_frame()

In [ ]:
# Seller risk: sellers with enough orders, ranked by late rate
seller_del = items_full.drop_duplicates(["order_id", "seller_id"]).merge(delivered[["order_id", "on_time", "review_score"]], on="order_id")
seller_perf = (seller_del.groupby("seller_id").agg(orders=("order_id", "count"), revenue=("item_revenue", "sum"),
    late_rate=("on_time", lambda s: 1 - s.mean()), avg_review=("review_score", "mean")).reset_index())
seller_perf = seller_perf[seller_perf["orders"] >= 30]

worst_cut = seller_perf["late_rate"].quantile(0.90)
worst = seller_perf[seller_perf["late_rate"] >= worst_cut]
print(f"Sellers analysed: {len(seller_perf)}  |  worst 10% late-rate threshold: {worst_cut:.1%}")
print(f"Worst 10% of sellers: {worst['revenue'].sum() / seller_perf['revenue'].sum():.1%} of revenue, average review {worst['avg_review'].mean():.2f} vs {seller_perf['avg_review'].mean():.2f} overall")
worst.sort_values("revenue", ascending=False).head(10)

## 11. Opportunities

In [ ]:
# High-revenue categories with below-average on-time rate: biggest payoff from fixing logistics
avg_ot = delivered["on_time"].mean()
opp = cat_big[(cat_big["on_time_rate"] < avg_ot)].sort_values("revenue", ascending=False).head(10)
print(f"Overall on-time rate: {avg_ot:.1%}")
display(opp)

# States with weak on-time rate and meaningful order volume
opp_state = state[(state["orders"] >= 300) & (state["on_time_rate"] < avg_ot)].sort_values("orders", ascending=False)
display(opp_state.head(8))

## 12. Estimating the cost of late delivery

In [ ]:
late = delivered[~delivered["on_time"]]
rev_late = late["revenue"].sum()
print(f"Revenue from late orders: {rev_late:,.0f} BRL ({rev_late / delivered['revenue'].sum():.1%} of delivered revenue)")

rr = first.groupby("on_time")["repeat"].mean()
late_customers = (~first["on_time"]).sum()
lost_repeat = late_customers * (rr[True] - rr[False])
avg_aov = df["revenue"].mean()
print(f"Repeat rate: on-time first order {rr[True]:.2%} vs late {rr[False]:.2%}")
print(f"If late-order customers repeated at the on-time rate: ~{max(lost_repeat, 0):,.0f} extra repeat customers")
print(f"Estimated revenue opportunity: ~{max(lost_repeat, 0) * avg_aov:,.0f} BRL (one extra order each at average order value)")

## 13. Key findings and recommendations
*Fill in the numbers from the outputs above before submitting.*

| # | Finding (evidence) | Recommended action | Expected impact |
|---|---|---|---|
| 1 | Orders 8+ days late average **X** stars vs **Y** for on-time orders (Section 8.1) | Set more realistic delivery estimates for remote states | Fewer 1-2 star reviews |
| 2 | States A, B, C have the lowest on-time rates (Section 8.2) | Add regional carriers or warehouses in these states | Higher on-time rate, better reviews |
| 3 | Worst 10% of sellers generate **Z%** of revenue but the lowest reviews (Section 10) | Performance plan, monthly tracking, suspension for repeat offenders | Fewer late orders |
| 4 | Repeat rate is **P%** after on-time vs **Q%** after late first orders (Section 9) | Coupon or apology for customers with late orders | Higher repeat rate |
| 5 | Top revenue categories with low on-time rate (Section 11) | Prioritise logistics fixes there first | Biggest revenue protection |

**Limitations:** data covers 2016-2018 only; delivery delays are measured against the platform's own estimate; correlation between delay and reviews does not prove causation.

In [ ]:

import os
os.makedirs("outputs", exist_ok=True)
kpi_df.to_csv("outputs/kpis.csv")
monthly.to_csv("outputs/monthly_trends.csv", index=False)
state.to_csv("outputs/state_performance.csv", index=False)
cat_big.to_csv("outputs/category_performance.csv", index=False)
worst.to_csv("outputs/worst_sellers.csv", index=False)
print("Saved to ./outputs")